In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *
from view import *
from reprojection import *


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])


#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,31),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

t = np.where(np.all([dates > datetime(2026,7,22),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260722T030009_V202610200250C_0647220501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260928T093003_V202610191523C_0649280504.fits'),
 212)

In [3]:
data, view = make_map(files[:],
                      correct_dr=True,
                      kind='synoptic',
                      correct_bias=True,
                      )

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:97: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))


In [5]:
show_data(data, view, vmin=-500, vmax=500)

(<Figure size 1500x600 with 2 Axes>, <Axes: >)

In [5]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2315.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.09.12_09:28:02_TAI'

In [6]:
show_data(data_, view, vmin=-500, vmax=500)

In [7]:
nx = len(data)
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
flux = np.nanmean(data, axis=1)

nx = len(data_)
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
flux_hmi = np.nanmean(data_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat, flux, label='FDT')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_295627/166170330.py:7: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)


In [8]:
np.mean(flux) / np.mean(np.abs(flux))

np.float64(0.01854587296313593)

In [9]:
np.savez('map.npz', data=data)